# Cài đặt thư viện cần thiết

In [1]:
!pip install -q pyvi emoji transformers scikit-learn openpyxl accelerate

import os
import json
import re
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModel, AutoConfig, get_scheduler
from sklearn.metrics import f1_score, classification_report
from pyvi.ViTokenizer import tokenize
import emoji
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

os.makedirs("/kaggle/working/saved_models", exist_ok=True)
os.makedirs("/kaggle/working/reports", exist_ok=True)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 54.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 46.5 MB/s eta 0:00:00
Device: cuda


# Khai báo thư viện và kiểm tra GPU

In [2]:
import subprocess

REPO_URL = "https://github.com/ricardo-tran/ViGoEmotions.git"
REPO_DIR = "/kaggle/working/ViGoEmotions_Original"

if not os.path.exists(REPO_DIR):
    print("Cloning repo...")
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    print("Repo already exists")

DOCS_PATH = os.path.join(REPO_DIR, "model", "docs")
CORPUS_PATH = os.path.join(REPO_DIR, "corpus")

# Load dictionaries
with open(os.path.join(DOCS_PATH, "patterns.json"), encoding="utf-8") as f:
    pattern_dict = json.load(f)

with open(os.path.join(DOCS_PATH, "emojis.json"), encoding="utf-8") as f:
    emoji_dict = json.load(f)

teen_dict = {}
with open(os.path.join(DOCS_PATH, "teencode4.txt"), encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line and "\t" in line:
            old, new = line.split("\t", 1)
            teen_dict[old] = new

print("✅ Dictionaries loaded")

# Load dataset
excel_path = os.path.join(CORPUS_PATH, "dataset_V1.xlsx")
excel_file = pd.ExcelFile(excel_path)

if "train" in excel_file.sheet_names:
    train_df = pd.read_excel(excel_file, sheet_name="train")
    val_df   = pd.read_excel(excel_file, sheet_name="val")
    test_df  = pd.read_excel(excel_file, sheet_name="test")
else:
    df = pd.read_excel(excel_file, sheet_name="Sheet1")
    train_df = df[df["set"] == "train"].copy()
    val_df   = df[df["set"] == "val"].copy()
    test_df  = df[df["set"] == "test"].copy()

print(f"Train: {train_df.shape} | Val: {val_df.shape} | Test: {test_df.shape}")

Cloning repo...


Cloning into '/kaggle/working/ViGoEmotions_Original'...


✅ Dictionaries loaded
Train: (16531, 3) | Val: (2066, 3) | Test: (2067, 3)


# Cấu hình đường dẫn và Tải Dữ liệu (Dictionaries & Dataset) 

In [3]:
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()

    # 1. Normalize patterns
    for pattern, replacement in pattern_dict.items():
        text = re.sub(pattern, replacement, text)

    # 2. Remove duplicate alphabet characters
    result, prev = [], None
    for char in text:
        if char.isalpha() and char == prev:
            continue
        prev = char
        result.append(char)
    text = "".join(result)

    # 3. Remove duplicate emojis
    result, prev_emoji = [], None
    for char in text:
        if char in emoji.EMOJI_DATA:
            if char == prev_emoji:
                continue
            prev_emoji = char
        else:
            prev_emoji = None
        result.append(char)
    text = "".join(result)

    # 4. Replace teencode
    for old, new in teen_dict.items():
        text = re.sub(rf"\b{re.escape(old)}\b", new, text)

    # 5. Replace emojis
    for emj, rep in emoji_dict.items():
        text = text.replace(emj, f" {rep} ")

    # 6. Format punctuation & whitespace
    text = re.sub(r"(?<![.,!?;:])\n", ". ", text)
    text = re.sub(r"\n([.,!?;:])?", r" \1", text)
    text = re.sub(r"([.,!?;:])", r" \1 ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

print("Applying S2 preprocessing...")
for df in [train_df, val_df, test_df]:
    df["text"] = df["text"].astype(str).apply(clean_text)

print("✅ Preprocessing done")
print(train_df["text"].iloc[0])

Applying S2 preprocessing...
✅ Preprocessing done
xem mà ngẫm lại cuộc đời bản thân ta đã trải qua nhiều thứ ta rồi cũng sẽ lớn kí ước sẽ còn mãi trong lòng


In [4]:
!find /kaggle/working/ViGoEmotions_Original -maxdepth 3 -type f

/kaggle/working/ViGoEmotions_Original/annotation/groq_annotator_llama_3_70b_public.ipynb
/kaggle/working/ViGoEmotions_Original/annotation/llm_guideline_official.md
/kaggle/working/ViGoEmotions_Original/annotation/google_annotator_gemma_3_public.ipynb
/kaggle/working/ViGoEmotions_Original/annotation/google_annotator_gemini_2_0f_public.ipynb
/kaggle/working/ViGoEmotions_Original/README.md
/kaggle/working/ViGoEmotions_Original/corpus/label_dict.json
/kaggle/working/ViGoEmotions_Original/corpus/train.csv
/kaggle/working/ViGoEmotions_Original/corpus/dataset_V1.xlsx
/kaggle/working/ViGoEmotions_Original/corpus/val.csv
/kaggle/working/ViGoEmotions_Original/corpus/test.csv
/kaggle/working/ViGoEmotions_Original/model/ViT5.ipynb
/kaggle/working/ViGoEmotions_Original/model/docs/patterns.json
/kaggle/working/ViGoEmotions_Original/model/docs/label_dict.json
/kaggle/working/ViGoEmotions_Original/model/docs/emojis.json
/kaggle/working/ViGoEmotions_Original/model/docs/teencode4.txt
/kaggle/working/ViG

# Tiền xử lý văn bản (S2 Preprocessing)

In [5]:
with open(os.path.join(DOCS_PATH, "label_dict.json"), encoding="utf-8") as f:
    label_dict = json.load(f)

label_to_idx = {label: int(idx) for idx, label in label_dict.items()}
print("Number of labels:", len(label_dict))

def encode_labels(label_str, label_dict):
    labels = str(label_str).replace("[", "").replace("]", "").replace("'", "").replace('"', "").split(",")
    labels = [x.strip() for x in labels if x.strip()]
    vec = np.zeros(len(label_dict), dtype=np.float32)

    if labels and labels[0].isnumeric():
        labels = [int(x) for x in labels]
        for idx in label_dict.values():
            if idx in labels:
                vec[idx] = 1.0
    else:
        for lab, idx in label_dict.items():
            if lab in labels:
                vec[idx] = 1.0
    return vec

train_texts  = train_df["text"].tolist()
train_labels = [encode_labels(x, label_to_idx) for x in train_df["labels"]]
val_texts    = val_df["text"].tolist()
val_labels   = [encode_labels(x, label_to_idx) for x in val_df["labels"]]
test_texts   = test_df["text"].tolist()
test_labels  = [encode_labels(x, label_to_idx) for x in test_df["labels"]]

print("✅ Labels encoded")
print("Example label:", train_labels[0])

Number of labels: 28
✅ Labels encoded
Example label: [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0.]


# Load Labels và Mã hóa One-hot (Label Encoding)

In [6]:
# ===== CHỌN MODEL BARTpho =====
model_type = "bartpho"
model_name = "vinai/bartpho-syllable-base"   # hoặc "vinai/bartpho-syllable" nếu muốn large
max_len = 200
BATCH_SIZE = 16                              # có thể giảm xuống 8 nếu vẫn chậm/OOM
# ==============================

tokenizer = AutoTokenizer.from_pretrained(model_name)

class SentimentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = torch.tensor(labels, dtype=torch.float32)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        
        # BARTpho dùng syllable, vẫn giữ clean_text S2, KHÔNG cần pyvi.tokenize thêm
        encoding = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_len,
            padding="max_length",
            return_tensors="pt",
            return_attention_mask=True,
        )
        return {
            "input_ids": encoding["input_ids"].flatten(),
            "attention_mask": encoding["attention_mask"].flatten(),
            "targets": self.labels[idx],
            "text": text,
        }

train_dataset = SentimentDataset(train_texts, train_labels, tokenizer, max_len)
val_dataset   = SentimentDataset(val_texts, val_labels, tokenizer, max_len)
test_dataset  = SentimentDataset(test_texts, test_labels, tokenizer, max_len)

# num_workers=0 để tránh treo
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print("✅ DataLoader ready (BARTpho)")
print("Train batches:", len(train_loader))

config.json:   0%|          | 0.00/898 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

dict.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

✅ DataLoader ready (BARTpho)
Train batches: 1034


/tmp/ipykernel_25/2219047049.py:13: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  self.labels = torch.tensor(labels, dtype=torch.float32)


# Khởi tạo Tokenizer và DataLoader

In [7]:
class ModelSentimentClassifier(nn.Module):
    def __init__(self, n_classes, model_name, model_type):
        super().__init__()
        self.model_type = model_type
        config = AutoConfig.from_pretrained(
            model_name,
            hidden_dropout_prob=0.1,
            attention_probs_dropout_prob=0.1
        )
        self.backbone = AutoModel.from_pretrained(model_name, config=config)
        self.drop = nn.Dropout(0.2)
        self.fc = nn.Linear(self.backbone.config.hidden_size, n_classes)

    def forward(self, input_ids, attention_mask):
        outputs = self.backbone(
            input_ids=input_ids,
            attention_mask=attention_mask,
            return_dict=True
        )
        # BARTpho không có pooler_output → lấy token đầu tiên
        if "bartpho" in self.model_type:
            pooled = outputs.last_hidden_state[:, 0, :]
        else:
            pooled = outputs.pooler_output if outputs.pooler_output is not None else outputs.last_hidden_state[:, 0, :]
        
        x = self.drop(pooled)
        return {"logits": self.fc(x)}

model = ModelSentimentClassifier(
    n_classes=len(label_dict),
    model_name=model_name,
    model_type=model_type
).to(device)

print("✅ BARTpho model loaded")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

pytorch_model.bin:   0%|          | 0.00/526M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/265 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


model.safetensors:   0%|          | 0.00/526M [00:00<?, ?B/s]

✅ BARTpho model loaded
Parameters: 193,070,620


# Cấu hình

In [8]:
EPOCHS = 12
optimizer = AdamW(model.parameters(), lr=5e-5)
lr_scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=len(train_loader),
    num_training_steps=len(train_loader) * EPOCHS
)

# pos_weight chống mất cân bằng
label_counts = np.sum(train_labels, axis=0)
pos_weight = torch.tensor(
    [(len(train_labels) - c) / max(c, 1) for c in label_counts],
    dtype=torch.float32
).to(device)
loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

def run_epoch(model, loader, is_train=True):
    model.train() if is_train else model.eval()
    losses, all_y, all_p = [], [], []

    context = torch.enable_grad() if is_train else torch.no_grad()
    with context:
        for batch in tqdm(loader, leave=False):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            targets = batch["targets"].to(device)

            if is_train:
                optimizer.zero_grad()

            logits = model(input_ids, attention_mask)["logits"]
            loss = loss_fn(logits, targets)
            losses.append(loss.item())

            if is_train:
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                lr_scheduler.step()

            preds = (torch.sigmoid(logits) >= 0.5).int()
            all_y.append(targets.cpu().numpy())
            all_p.append(preds.cpu().numpy())

    y = np.vstack(all_y)
    p = np.vstack(all_p)
    f1 = f1_score(y, p, average="macro", zero_division=0)
    return np.mean(losses), f1

best_f1 = 0.0
history = {"train_loss": [], "train_f1": [], "val_loss": [], "val_f1": []}

print("Start training...")
for epoch in range(1, EPOCHS + 1):
    print(f"\n===== Epoch {epoch}/{EPOCHS} =====")
    train_loss, train_f1 = run_epoch(model, train_loader, is_train=True)
    val_loss, val_f1 = run_epoch(model, val_loader, is_train=False)

    history["train_loss"].append(train_loss)
    history["train_f1"].append(train_f1)
    history["val_loss"].append(val_loss)
    history["val_f1"].append(val_f1)

    print(f"Train Loss: {train_loss:.4f} | Train Macro-F1: {train_f1:.4f}")
    print(f"Val   Loss: {val_loss:.4f} | Val   Macro-F1: {val_f1:.4f}")

    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(model.state_dict(), f"/kaggle/working/saved_models/{model_type}_best.pth")
        print(f"⭐ Saved best model (Val F1 = {best_f1:.4f})")

print("\n✅ Training finished!")

Start training...

===== Epoch 1/12 =====



100%|██████████| 1034/1034 [11:44<00:00,  1.87it/s]


Train Loss: 1.0785 | Train Macro-F1: 0.2200
Val   Loss: 0.8181 | Val   Macro-F1: 0.3792
⭐ Saved best model (Val F1 = 0.3792)

===== Epoch 2/12 =====


Train Loss: 0.7384 | Train Macro-F1: 0.3821
Val   Loss: 0.7309 | Val   Macro-F1: 0.3980
⭐ Saved best model (Val F1 = 0.3980)

===== Epoch 3/12 =====


Train Loss: 0.5800 | Train Macro-F1: 0.4579
Val   Loss: 0.7631 | Val   Macro-F1: 0.4396
⭐ Saved best model (Val F1 = 0.4396)

===== Epoch 4/12 =====


...

Train Loss: 0.1797 | Train Macro-F1: 0.7679
Val   Loss: 1.3876 | Val   Macro-F1: 0.5319
⭐ Saved best model (Val F1 = 0.5319)

===== Epoch 9/12 =====


Train Loss: 0.1452 | Train Macro-F1: 0.8107
Val   Loss: 1.5137 | Val   Macro-F1: 0.5327
⭐ Saved best model (Val F1 = 0.5327)

===== Epoch 10/12 =====


Train Loss: 0.1163 | Train Macro-F1: 0.8478
Val   Loss: 1.6863 | Val   Macro-F1: 0.5359
⭐ Saved best model (Val F1 = 0.5359)

===== Epoch 11/12 =====


Train Loss: 0.0932 | Train Macro-F1: 0.8759
Val   Loss: 1.7325 | Val   Macro-F1: 0.5395
⭐ Saved best model (Val F1 = 0.5395)

===== Epoch 12/12 =====


Train Loss: 0.0787 | Train Macro-F1: 0.8965
Val   Loss: 1.7942 | Val   Macro-F1: 0.5424
⭐ Saved best model (Val F1 = 0.5424)

✅ Training finished!


# Thiết lập Hàm Huấn luyện & Đánh giá

In [9]:
# Load best model
model.load_state_dict(torch.load(f"/kaggle/working/saved_models/{model_type}_best.pth"))
model.eval()

all_targets, all_preds = [], []

with torch.no_grad():
    for batch in tqdm(test_loader):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        targets = batch["targets"].to(device)

        logits = model(input_ids, attention_mask)["logits"]
        preds = (torch.sigmoid(logits) >= 0.5).int()

        all_targets.append(targets.cpu().numpy())
        all_preds.append(preds.cpu().numpy())

y_true = np.vstack(all_targets)
y_pred = np.vstack(all_preds)

macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
micro_f1 = f1_score(y_true, y_pred, average="micro", zero_division=0)

print("\n========== TEST RESULTS ==========")
print(f"Macro F1: {macro_f1:.4f}")
print(f"Micro F1: {micro_f1:.4f}")
print("\nClassification Report:")
print(classification_report(
    y_true, y_pred,
    target_names=list(label_dict.values()),
    zero_division=0
))

100%|██████████| 130/130 [00:29<00:00,  4.34it/s]


========== TEST RESULTS ==========
Macro F1: 0.5560
Micro F1: 0.5768

Classification Report:
                precision    recall  f1-score   support

     amusement       0.66      0.80      0.72       374
    excitement       0.40      0.51      0.45        98
           joy       0.50      0.64      0.56       204
          love       0.57      0.82      0.67       143
        desire       0.32      0.51      0.39        80
      optimism       0.58      0.73      0.65       142
        caring       0.55      0.67      0.60       150
         pride       0.63      0.71      0.67        86
    admiration       0.46      0.59      0.52       101
     gratitude       0.78      0.89      0.83       108
        relief       0.39      0.63      0.48        60
      approval       0.52      0.73      0.61       115
   realization       0.34      0.38      0.36        95
      surprise       0.43      0.53      0.48        85
     curiosity       0.52      0.69      0.59       100
     conf

# train loop